In [1]:
# !pip install -qU torch
# !pip install -q transformers==4.44.1
# !pip install -q accelerate
# !pip install -q bitsandbytes==0.43.3
# !pip install -q datasets==2.21.0
# !pip install -q trl==0.9.6
# !pip install -q peft==0.12.0
# !pip install -qU "huggingface_hub[cli]"
# dbutils.library.restartPython()

In [0]:
import os
import torch
import pandas as pd

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig
from transformers import TrainingArguments

# from transformers import pipeline
from transformers import logging
from transformers import HfArgumentParser
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)

from datasets import load_dataset
from datasets import Dataset

from trl import SFTTrainer, setup_chat_format

2025-01-22 10:27:37.193429: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-22 10:27:37.228355: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[2025-01-22 10:27:40,604] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.5
 [WARNING]  using untested triton version (3.1.0), only 1.0.0 is known to be compatible


/databricks/python/lib/python3.11/site-packages/deepspeed/runtime/zero/linear.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @autocast_custom_fwd
/databricks/python/lib/python3.11/site-packages/deepspeed/runtime/zero/linear.py:66: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @autocast_custom_bwd


### Define constants

In [0]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
dataset_name = "ruslanmv/ai-medical-chatbot"
qlora_model_path = "models/lora/llama31_8B_chat_doctor/"
hf_token = "<your-token-here>"

### Login to HF Hub

In [0]:
from huggingface_hub import login

login(token=hf_token)

### Set data type and attention mechanism

In [0]:
torch_dtype = torch.float16
attn_implementation = "eager"

### Load Model

In [0]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation,
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### Load tokenizer

In [0]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model, tokenizer = setup_chat_format(model, tokenizer)

### Evaluate the pre-trained model

In [0]:
messages = [
    {
        "role": "user",
        "content": "Hello doctor, I have bad acne. How do I get rid of it?",
    }
]

prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(
    "cuda"
)

outputs = model.generate(**inputs, max_length=150, num_return_sequences=1)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text.split("assistant")[1])


I can't provide medical advice, but I can offer some general information about acne. Would that help?


### Adding the adapter to the layer

In [0]:
# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "up_proj",
        "down_proj",
        "gate_proj",
        "k_proj",
        "q_proj",
        "v_proj",
        "o_proj",
    ],
)
model = get_peft_model(model, peft_config)

### Load dataset

In [0]:
dataset = load_dataset(dataset_name, split="all")
dataset = dataset.shuffle(seed=65).select(
    range(1000)
)  # Only use 1000 samples for quick demo
dataset[0]

/databricks/python_shell/dbruntime/huggingface_patches/datasets.py:45: UserWarning: The cache_dir for this dataset is /root/.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [DBFS].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/dbfs/']
  warnings.warn(warning_message)
/databricks/python_shell/dbruntime/huggingface_patches/datasets.py:14: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


{'Description': 'Can blood pressure medication be stopped to check improvement in bp levels?',
 'Patient': "I'm 35, BP 150/100 without medicine, never smoke, drink, BMIMy hdl:45, LDL:107, Total Cholesterol:168, Triglyceride:95, Blood Sugar(Fasting):83 (all are without medicine) My question is, should I stop this medicine to see the afffect. it is worth to mention that recently I've increased my physical activity,",
 'Doctor': 'Hello,Thanks for writing to Health Care Magic, I am Dr Asad Riaz, I have closely read your question and I understand your concerns, I will hereby guide you regarding your health related problem.BP is one major risk factor for major complication like MI or stroke...if pt have high BP as u r having ..1st we need to do life style modification to see whteher it work to de BP or not in whch i advice pt to lower salt intake,drinkng dec weight n exercise for a period of 6 month n still if pt have high BP then need to add med to control it so we prevent major comlication

### Format chat template

In [0]:
def format_chat_template(row):
    row_json = [
        {"role": "user", "content": row["Patient"]},
        {"role": "assistant", "content": row["Doctor"]},
    ]

    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

In [0]:
print(format_chat_template(dataset[0])["text"])

<|im_start|>user
I'm 35, BP 150/100 without medicine, never smoke, drink, BMIMy hdl:45, LDL:107, Total Cholesterol:168, Triglyceride:95, Blood Sugar(Fasting):83 (all are without medicine) My question is, should I stop this medicine to see the afffect. it is worth to mention that recently I've increased my physical activity,<|im_end|>
<|im_start|>assistant
Hello,Thanks for writing to Health Care Magic, I am Dr Asad Riaz, I have closely read your question and I understand your concerns, I will hereby guide you regarding your health related problem.BP is one major risk factor for major complication like MI or stroke...if pt have high BP as u r having ..1st we need to do life style modification to see whteher it work to de BP or not in whch i advice pt to lower salt intake,drinkng dec weight n exercise for a period of 6 month n still if pt have high BP then need to add med to control it so we prevent major comlication..as u having high bp u dont need to stop med n if u wanna it visit ur ph

In [0]:
dataset = dataset.map(
    format_chat_template,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

### Split train test

In [0]:
dataset = dataset.train_test_split(test_size=0.1)

In [0]:
len(dataset["train"]), len(dataset["test"])

(900, 100)

### Define training Arguments

In [0]:
qlora_model_path = "/dbfs/qlora_model"

In [0]:
training_arguments = TrainingArguments(
    output_dir=qlora_model_path,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    evaluation_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    group_by_length=True,
)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c8ffc70-8b1e-4252-929b-360b79bc5a09/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


### Define SFTTrainer

In [0]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    max_seq_length=512,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

/databricks/python/lib/python3.11/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c8ffc70-8b1e-4252-929b-360b79bc5a09/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c8ffc70-8b1e-4252-929b-360b79bc5a09/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/local_disk0/.ephemeral_nfs/env

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

[rank0]:[W122 10:28:54.149117536 ProcessGroupNCCL.cpp:4115] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 to perform barrier as devices used by this process are currently unknown. This can potentially cause a hang if this rank to GPU mapping is incorrect.Specify device_ids in barrier() to force use of a particular device,or call init_process_group() with a device_id.


### Start training

In [0]:
trainer.train()

[rank0]:[W122 10:29:03.206231966 reducer.cpp:1400] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())


Step,Training Loss,Validation Loss
90,2.573300,2.521845
180,2.383500,2.477083
270,2.591500,2.445400
360,2.132500,2.424166
450,2.602400,2.411577


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c8ffc70-8b1e-4252-929b-360b79bc5a09/lib/python3.11/site-packages/peft/utils/save_and_load.py:232: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=450, training_loss=2.5035127719243366, metrics={'train_runtime': 341.5385, 'train_samples_per_second': 2.635, 'train_steps_per_second': 1.318, 'total_flos': 9291079669514240.0, 'train_loss': 2.5035127719243366, 'epoch': 1.0})

### Evaluate the model

In [0]:
messages = [
    {
        "role": "user",
        "content": "Hello doctor, I have bad acne. How do I get rid of it?",
    }
]

prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

In [0]:
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(
    "cuda"
)

outputs = model.generate(**inputs, max_length=150, num_return_sequences=1)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text.split("assistant")[1])


Hi. For more information consult a dermatologist online -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->


### Save the trained model

In [0]:
trainer.model.save_pretrained(qlora_model_path)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c8ffc70-8b1e-4252-929b-360b79bc5a09/lib/python3.11/site-packages/peft/utils/save_and_load.py:232: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


### Load and merge the adapter

In [0]:
qlora_model_path

'/dbfs/qlora_model'

In [0]:
dbutils.fs.ls("dbfs:/qlora_model")

[FileInfo(path='dbfs:/qlora_model/README.md', name='README.md', size=5111, modificationTime=1737542114000),
 FileInfo(path='dbfs:/qlora_model/adapter_config.json', name='adapter_config.json', size=740, modificationTime=1737542119000),
 FileInfo(path='dbfs:/qlora_model/adapter_model.safetensors', name='adapter_model.safetensors', size=2269211544, modificationTime=1737542119000),
 FileInfo(path='dbfs:/qlora_model/checkpoint-450/', name='checkpoint-450/', size=0, modificationTime=1737542077000),
 FileInfo(path='dbfs:/qlora_model/runs/', name='runs/', size=0, modificationTime=1737541743000)]

In [0]:
from peft import PeftModel

base_model_reload = AutoModelForCausalLM.from_pretrained(
    model_name,
    return_dict=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

base_model_reload, tokenizer = setup_chat_format(base_model_reload, tokenizer)

# Merge adapter with base model
model = PeftModel.from_pretrained(base_model_reload, qlora_model_path)

model = model.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [0]:
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(
    "cuda"
)

outputs = model.generate(**inputs, max_length=150, num_return_sequences=1)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text.split("assistant")[1])


Hello,Welcome to HCM. I have reviewed your query and here is my advice. I would suggest you to consult a dermatologist. If the acne is mild, it can be treated with topical antibiotics. In case of severe acne, systemic antibiotics like doxycycline or minocycline can be used. For more information consult a dermatologist online -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  -->  --> 


In [0]:
# from peft import AutoPeftModelForCausalLM
# from transformers import BitsAndBytesConfig

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch_dtype,
#     bnb_4bit_use_double_quant=True,
# )


# loaded_model = AutoPeftModelForCausalLM.from_pretrained(
#                                         qlora_model_path,
#                                         quantization_config = bnb_config,
#                                         device_map = 'auto')